# 5차시 (3) BERT 한국어 문장 분류 — 함께 라벨링하고 직접 학습하기

한경국립대 2026 여름특강 · 딥러닝/머신러닝 입문 (초급반)

이번 노트북에서는 **우리가 직접 만든 데이터** 로 한국어 문장 분류 모델을 학습합니다.

1. **다 같이 라벨링** — Google Sheet에 문장과 정답(label)을 함께 적습니다.
2. **데이터 불러오기** — 시트를 CSV로 내려받아 `pandas` 로 읽습니다.
3. **BERT 파인튜닝** — Hugging Face의 한국어 BERT(`klue/bert-base`)를 우리 데이터로 추가 학습합니다.
4. **예측** — 새 문장을 넣어 모델이 분류하는지 확인합니다.

4차시에서는 scikit-learn으로 머신러닝 **분류**(붓꽃 품종 맞히기)를 직접 만들어 봤습니다. 이번에는 같은 '분류' 문제를 **딥러닝 모델(BERT)** 로 풀어 봅니다. 표 형태의 숫자 특징 대신 **한국어 문장 자체**를 입력으로 받아, 문장의 의미를 이해하는 딥러닝의 힘으로 분류 성능을 한 단계 끌어올리는 것이 목표입니다.

> **BERT** 는 문장의 의미를 이해하도록 미리 학습된 모델입니다. 여기에 우리 데이터를 조금 더 학습시키면(=**파인튜닝**) 우리 문제에 맞는 분류기가 됩니다.

## ⚠️ 시작 전 필수: GPU 런타임 켜기

모델 학습은 **GPU** 가 있어야 빠릅니다. 다음 순서로 켜 주세요.

1. 상단 메뉴 **[런타임] → [런타임 유형 변경]**
2. **하드웨어 가속기** 를 **T4 GPU** 로 선택 → **저장**
3. 런타임이 다시 연결되면, 아래 셀로 GPU가 잡혔는지 확인합니다.

(GPU가 없어도 학습은 되지만 몇 배 느립니다. 꼭 켜 주세요.)

In [ ]:
import torch

if torch.cuda.is_available():
    print("GPU 사용 가능:", torch.cuda.get_device_name(0))
else:
    print("GPU가 꺼져 있습니다. [런타임]>[런타임 유형 변경]>T4 GPU 로 설정 후 다시 실행하세요.")

### 라이브러리 설치
Hugging Face의 `transformers`(모델), `datasets`(데이터 관리), `evaluate`(정확도 측정)를 설치합니다.
Colab에는 `torch`·`pandas` 가 이미 있습니다.

In [ ]:
!pip install -qU transformers datasets evaluate
print("설치 완료!")

---
# Part A · 다 같이 데이터 라벨링하기 (Google Sheet)

좋은 분류기를 만들려면 **라벨(정답)이 달린 데이터** 가 필요합니다. 오늘은 수강생이 **함께 Google Sheet 한 장** 을 채웁니다.

## A-1. 라벨링 주제 정하기

반에서 주제를 **하나** 고릅니다(강사가 안내). 예시:

| 주제 | label 값 | 문장 예시 |
|---|---|---|
| **영화/상품 리뷰 감정** | `긍정` / `부정` | "연기가 정말 최고였어요" → 긍정 / "시간이 아까웠다" → 부정 |
| **문의 의도 분류** | `배송` / `환불` / `상품문의` | "택배 언제 와요?" → 배송 / "환불하고 싶어요" → 환불 |
| **뉴스 카테고리** | `스포츠` / `경제` / `연예` | "손흥민 결승골" → 스포츠 |

> 처음엔 **2개 클래스(긍정/부정)** 가 가장 쉽고 결과도 잘 나옵니다. 클래스마다 **최소 20~30문장** 을 모으는 것을 목표로 합니다.

## A-2. Google Sheet 만들고 양식 맞추기

1. 브라우저에서 새 Google Sheet 를 엽니다 → 주소창에 **`sheet.new`** 입력하면 바로 생성됩니다.
2. **첫 번째 줄(헤더)** 을 정확히 이렇게 적습니다. 열 이름은 영어 소문자로:

   | A열 | B열 |
   |---|---|
   | **text** | **label** |

3. 둘째 줄부터 한 줄에 하나씩, 문장과 정답을 적습니다.

   | text | label |
   |---|---|
   | 이 영화 진짜 재밌어요 | 긍정 |
   | 스토리가 너무 지루했다 | 부정 |
   | 배우 연기가 인상 깊었어요 | 긍정 |

**주의사항**
- `label` 값은 **오타 없이 통일** 하세요. `긍정`/`긍정 `(뒤 공백)/`positive` 가 섞이면 다른 클래스로 취급됩니다.
- 문장(`text`)에 쉼표가 있어도 괜찮습니다(셀 한 칸이면 됩니다).

## A-3. 시트를 '공개 공유'로 바꾸기

노트북이 시트를 읽으려면 **링크가 있는 사람은 보기 가능** 상태여야 합니다.

1. 오른쪽 위 **[공유]** 버튼 클릭
2. **일반 액세스** 를 **[제한됨]** → **[링크가 있는 모든 사용자]** 로 변경
3. 권한은 **[뷰어]** 로 두면 충분합니다 → **완료**

> 이렇게 하면 링크를 아는 사람만 *볼 수* 있고, 수정은 막힙니다(안전).
> 반 전체가 **같은 시트를 함께 편집** 하려면, 편집은 강사가 학생 계정을 편집자로 추가하거나 "링크가 있는 모든 사용자"를 **편집자** 로 두면 됩니다(수업 중에만 권장).

## A-4. CSV 내려받기 주소(export URL) 만들기

Google Sheet 는 주소를 살짝 바꾸면 **CSV 로 바로 내려받기** 가 됩니다. `pandas` 가 이 주소를 직접 읽을 수 있습니다.

시트 주소는 보통 이렇게 생겼습니다:
```
https://docs.google.com/spreadsheets/d/[시트ID]/edit#gid=0
```
여기서 `[시트ID]` 부분(긴 영문/숫자)과 `gid` 숫자만 빼내어 아래처럼 만듭니다:
```
https://docs.google.com/spreadsheets/d/[시트ID]/export?format=csv&gid=0
```

아래 셀에 **시트ID** 와 **gid** 만 채우면 자동으로 주소를 만들어 줍니다.

In [ ]:
# TODO: 본인(또는 반 공용) 시트의 ID 와 gid 를 채우세요.
#   - SHEET_ID : 주소의 /d/ 와 /edit 사이의 긴 문자열
#   - GID      : 주소 끝 gid=0 의 숫자 (보통 0)
SHEET_ID = "여기에_시트ID_붙여넣기"
GID = "0"

csv_url = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv&gid={GID}"
print("CSV 주소:", csv_url)

## A-5. 데이터 불러오기 (pandas)

만든 주소를 `pd.read_csv` 로 읽으면 표(DataFrame)가 됩니다.

> 만약 시트가 아직 준비 안 됐거나 에러가 나면, **다음 셀의 '예비 데이터'** 로 먼저 진행할 수 있습니다(아래 A-6).

In [ ]:
import pandas as pd

try:
    df = pd.read_csv(csv_url)
    df = df.dropna(subset=["text", "label"])          # 빈 줄 제거
    df["text"] = df["text"].astype(str).str.strip()
    df["label"] = df["label"].astype(str).str.strip()  # label 앞뒤 공백 정리(오타 방지)
    print("시트에서 불러오기 성공! 데이터 수:", len(df))
    print("클래스별 개수:\n", df["label"].value_counts())
    USE_SHEET = True
except Exception as e:
    print("시트를 불러오지 못했습니다. (아직 준비 전이거나 공유 설정 확인 필요)")
    print("오류:", e)
    print("=> 아래 A-6 의 예비 데이터로 진행하세요.")
    USE_SHEET = False

## A-6. (예비) 시트가 없을 때 — 내장 예시 데이터

시트가 준비되지 않았거나 시간이 부족하면, 아래 **작은 감정 분류 데이터** 로 실습을 이어갈 수 있습니다.
위 A-5 가 성공했다면 **이 셀은 건너뛰어도** 됩니다(실행해도 시트 데이터를 덮어쓰지 않습니다).

In [ ]:
import pandas as pd

if not USE_SHEET:
    backup = [
        ("이 영화 정말 재미있었어요", "긍정"), ("연기가 너무 자연스럽고 좋았다", "긍정"),
        ("스토리가 탄탄해서 몰입했어요", "긍정"), ("음악이 분위기와 잘 어울렸다", "긍정"),
        ("기대 이상으로 만족스러웠습니다", "긍정"), ("다시 보고 싶은 작품이에요", "긍정"),
        ("배우들의 호흡이 환상적이었다", "긍정"), ("끝까지 긴장감이 넘쳤어요", "긍정"),
        ("가족과 보기 좋은 따뜻한 영화", "긍정"), ("여운이 오래 남는 명작이다", "긍정"),
        ("시간이 너무 아까웠어요", "부정"), ("내용이 지루하고 늘어졌다", "부정"),
        ("연기가 어색해서 몰입이 안 됐다", "부정"), ("결말이 너무 허무했어요", "부정"),
        ("기대했는데 실망스러웠습니다", "부정"), ("스토리가 산만하고 엉성했다", "부정"),
        ("두 번 다시 보고 싶지 않아요", "부정"), ("돈이 아까운 영화였다", "부정"),
        ("전개가 뻔하고 식상했어요", "부정"), ("소리만 시끄럽고 알맹이가 없다", "부정"),
    ]
    df = pd.DataFrame(backup, columns=["text", "label"])
    print("예비 데이터 사용. 데이터 수:", len(df))
    print(df["label"].value_counts())
else:
    print("시트 데이터를 사용 중이라 예비 데이터는 건너뜁니다.")

---
# Part B · 데이터 준비 (라벨 → 숫자, 학습/평가 분리)

모델은 글자 label(`긍정`/`부정`)을 바로 못 읽으므로 **숫자 id** 로 바꿉니다(`긍정`→0, `부정`→1 처럼).
그리고 데이터를 **학습용 80% / 평가용 20%** 로 나눕니다(학습에 안 쓴 데이터로 성적을 매기기 위해).

In [ ]:
# 1) label 글자 <-> 숫자 id 매핑표 만들기
label_names = sorted(df["label"].unique())
label2id = {name: i for i, name in enumerate(label_names)}
id2label = {i: name for name, i in label2id.items()}
print("클래스:", label2id)

# 2) label 을 숫자로 변환한 컬럼 추가
df["label_id"] = df["label"].map(label2id)

# 3) 학습/평가 분리 (stratify: 클래스 비율 유지)
from sklearn.model_selection import train_test_split
train_df, eval_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["label_id"]
)
print(f"학습 {len(train_df)}개 / 평가 {len(eval_df)}개")

Hugging Face `datasets` 형식으로 바꾸고, **토크나이저** 로 문장을 모델이 읽는 숫자(token)로 변환합니다.
토크나이저는 우리가 쓸 모델(`klue/bert-base`)과 **짝** 인 것을 써야 합니다.

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "klue/bert-base"   # 한국어로 사전학습된 BERT
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# pandas -> Dataset
train_ds = Dataset.from_pandas(train_df[["text", "label_id"]], preserve_index=False)
eval_ds = Dataset.from_pandas(eval_df[["text", "label_id"]], preserve_index=False)

# 문장을 토큰으로 변환하는 함수
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=64)

train_ds = train_ds.map(tokenize, batched=True).rename_column("label_id", "labels")
eval_ds = eval_ds.map(tokenize, batched=True).rename_column("label_id", "labels")
print("토큰화 완료. 예시 키:", train_ds.column_names)

---
# Part C · BERT 파인튜닝 (모델 학습)

이제 사전학습된 BERT를 불러와 **우리 클래스 수에 맞는 분류기** 를 붙이고, 학습합니다.
- `AutoModelForSequenceClassification` : BERT 위에 '분류용 출력층' 을 자동으로 얹어 줍니다.
- `num_labels` : 우리 클래스 개수.

처음 실행 시 모델 파일(수백 MB)을 내려받느라 잠시 걸립니다.

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
)
print("모델 준비 완료. 분류할 클래스 수:", len(label_names))

**정확도(accuracy)** 를 평가 지표로 쓰고, 학습 설정(`TrainingArguments`)을 정한 뒤 `Trainer` 로 학습합니다.
- `num_train_epochs` : 데이터를 몇 번 반복 학습할지. 데이터가 적으니 3~5회면 충분합니다.
- 데이터가 적으면 학습은 **1~3분** 내로 끝납니다(GPU 기준).

In [ ]:
import numpy as np
import evaluate
from transformers import TrainingArguments, Trainer

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=preds, references=labels)

args = TrainingArguments(
    output_dir="bert_classifier",
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    eval_strategy="epoch",     # 매 epoch 마다 평가
    logging_steps=10,
    report_to="none",          # 외부 로깅 끔
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

학습이 끝났습니다. 평가 데이터로 **최종 정확도** 를 확인합니다.

> 데이터가 적으면 정확도가 들쭉날쭉할 수 있습니다. 문장을 더 모으면 보통 올라갑니다.

In [ ]:
result = trainer.evaluate()
print("평가 정확도(accuracy):", round(result["eval_accuracy"], 3))

---
# Part D · 새 문장으로 예측해 보기

학습한 모델로 **처음 보는 문장** 을 분류해 봅시다.
`pipeline` 을 쓰면 토큰화·예측·후처리를 한 번에 해 줍니다.

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1,
)

test_sentences = [
    "이건 정말 인생 영화예요",
    "보다가 졸았습니다",
    "생각보다 괜찮았어요",
]
for s in test_sentences:
    out = classifier(s)[0]
    print(f"'{s}' -> {out['label']} (확신도 {out['score']:.2f})")

**TODO.** 아래에 여러분의 문장을 넣어 모델이 어떻게 분류하는지 시험해 보세요.
어떤 문장에서 틀리나요? 그런 문장을 라벨링 데이터에 더 넣으면 모델이 좋아집니다.

In [ ]:
# TODO: 분류해 보고 싶은 문장을 자유롭게 넣어 보세요.
my_sentences = [
    "",  # 예: "이 가격에 이 정도면 훌륭하죠"
]
for s in my_sentences:
    if s.strip():
        out = classifier(s)[0]
        print(f"'{s}' -> {out['label']} (확신도 {out['score']:.2f})")

---
## (심화·선택) 이 분류기를 챗봇의 '의도 분류기' 로 쓰기

방금 만든 분류기는 다음 노트북(멀티 에이전트)의 **의도 분류(라우팅)** 에 그대로 재활용할 수 있습니다.
예를 들어 label 을 `배송`/`환불`/`상품문의` 로 학습했다면, 사용자의 말이 어떤 의도인지 분류해 **알맞은 처리로 보내는** 라우터가 됩니다.

아래는 분류 결과(의도)에 따라 다른 안내를 돌려주는 **아주 간단한 라우터** 예시입니다.
(label 이 감정 분류라면 의도 라우팅 예시로는 어색할 수 있습니다 — 의도 데이터로 학습했을 때 더 잘 맞습니다.)

In [ ]:
def route_by_intent(message):
    intent = classifier(message)[0]["label"]   # 모델이 예측한 의도(label)
    # 의도별로 다른 처리를 연결(여기서는 안내 문구만)
    handlers = {
        "배송": "[배송팀] 주문번호를 알려주시면 배송 상태를 확인해 드릴게요.",
        "환불": "[환불팀] 환불 규정을 안내해 드리겠습니다. 영수증이 있으신가요?",
        "상품문의": "[상담팀] 어떤 상품이 궁금하신가요? 자세히 안내해 드릴게요.",
    }
    return handlers.get(intent, f"(의도: {intent}) 담당자에게 연결해 드릴게요.")

print(route_by_intent("제 택배 지금 어디쯤 왔나요?"))
print(route_by_intent("이거 환불 가능한가요?"))

---
## 마무리

- 우리가 **직접 라벨링한 데이터**(Google Sheet)로 한국어 BERT(`klue/bert-base`)를 **파인튜닝** 했습니다.
- 흐름: **데이터 수집·라벨링 → 숫자 변환 → 학습/평가 분리 → 토큰화 → 파인튜닝 → 예측**.
- 데이터가 많고 라벨이 깔끔할수록 정확도가 올라갑니다("데이터가 곧 성능").
- 이 분류기는 다음 노트북의 **의도 분류 라우터** 로도 쓸 수 있습니다.

> 학습한 모델을 저장하려면 `trainer.save_model("my_bert")` 로 저장하고, 나중에 `pipeline(model="my_bert")` 로 다시 불러올 수 있습니다.